# SpaceInvasion GA + Neural Network Optimizer — Interactive Experimentation Lab

Welcome to the **read-only experimentation and tuning handbook** for the `spaceinvasion` **Genetic Algorithm + Neural Network (`ga_nn`)** planner and **Topological Action Graph (DAG)** engine.

Every cell in this notebook runs **100% offline and side-effect-free**: it never opens a browser connection, never mutates `data/attack_history.json`, and inspects any live SQLite database (`state/spaceinvasion.db`) strictly through an isolated temporary copy.

---

### End-to-End Decision Pipeline

```text
  +----------------------------+      +-------------------------------+
  | Complete History Telemetry | ---> | 46D Empire + History Encoder  |
  | (JSON + Read-Only SQLite)  |      |   encode_empire_state(...)    |
  +----------------------------+      +---------------+---------------+
                                                      |
                                                      v
  +----------------------------+      +-------------------------------+
  | Online Value MLP Evaluator | <--- | Forward Horizon Rollout Sim   |
  | PurePythonMLP / TorchMLP   |      | simulate_action_sequence(...) |
  +-------------+--------------+      +---------------+---------------+
                |                                     |
                +------------------+------------------+
                                   |
                                   v
                +-------------------------------------+
                | Genetic Algorithm Chromosome Search |
                | run_optimizer / optimize_plan_...   |
                +------------------+------------------+
                                   |
                                   v
                +-------------------------------------+
                | Kahn Topological Action Graph (DAG) |
                | build_action_graph + Queue ETAs     |
                +-------------------------------------+
```

### Notebook Contents

| Chapter | Focus | Key Symbols |
|---|---|---|
| **0. Bootstrap** | CWD-safe path resolution, runtime capabilities, ASCII table/bar helpers | `resolve_repo_root`, `render_table`, `ascii_bar` |
| **1A. Attack History JSON** | Merchant parity (`6 : 2.5 : 3 : 1`), defensive scalar coercion & JSON audit | `load_attack_history_stats`, `_safe_float`, `_safe_int` |
| **1B. Read-Only SQLite WAL** | 4-layer temp-copy preflight & PII/token-redacted live DB inspection | `inspect_sqlite_readonly`, `GameDatabase` |
| **2A. Empire Sandbox** | SpaceInvasion Classic prerequisite-legal midgame colony & hourly economy | `build_sample_game_state`, `estimate_hourly_economy` |
| **2B. Multi-Table History Lab** | Controlled **Farm (`90%` win)** vs **Fluke Trap (`10%` win)** vs **Spacia** seed | `get_raid_stats`, `get_full_history_summary` |
| **3A. 46D State Encoding** | Cold-start vs complete-history feature vector deltas (`[0..45]`) | `encode_empire_state`, `STATE_FEATURE_DIM` |
| **3B. Rollouts & Value Net** | Multi-step candidate rollouts, online MLP training & `dV/dx_j` sensitivities | `simulate_action_sequence`, `NeuralValueNetwork` |
| **4A. GA + NN Evolution** | Chromosome search, generation convergence curve & fitness breakdown | `run_optimizer`, `generation_history` |
| **4B. Rollout Trace & ETAs** | Step-by-step best chromosome trace, `priority` vs `ga_nn` plans & wait ETAs | `attach_action_etas`, `compute_action_eta` |
| **5. Action Graph (DAG)** | Kahn topological layering, 11 edge classes & isolated config simulation | `build_action_graph`, `merge_strategy_config_overrides` |
| **6. Solar Policy Ablation** | Planetary solar ban (`DEMOLISH_BUILDING`) vs late-game field preservation | `StrategyConfig(forbid_planetary_solar=...)` |
| **7. Hyperparameter Sweep** | Interactive grid across `miner`, `hybrid`, and `fleeter` GA/NN configurations | `EXPERIMENT_CONFIGS` |

In [ ]:
import contextlib
from dataclasses import replace
import hashlib
import json
import math
from os import environ
from pathlib import Path
import shutil
import sqlite3
import sys
import tempfile
import time
from typing import Any


def resolve_repo_root() -> Path:
    """Locate the repository root from any working directory without bare `__file__`."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "spaceinvasion" / "__init__.py").is_file():
            return cand
    for entry in list(sys.path):
        if not entry:
            continue
        p = Path(entry).resolve()
        if (p / "spaceinvasion" / "__init__.py").is_file():
            return p
    import spaceinvasion  # raises ImportError if project is unreachable

    return Path(spaceinvasion.__file__).resolve().parents[1]


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer isolated `cwd` data/state fixtures when present; otherwise use `repo_root`."""
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "attack_history.json").is_file() or (
        cwd / "state" / "spaceinvasion.db"
    ).is_file():
        return cwd
    return repo_root


ACTIVE_DATA_ROOT = resolve_active_data_root(REPO_ROOT)
_LEGACY_ATTACK_JSON = ACTIVE_DATA_ROOT / "data" / "attack_history.json"
LIVE_DB_PATH = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
ATTACK_HISTORY_PATH = _LEGACY_ATTACK_JSON if _LEGACY_ATTACK_JSON.is_file() else LIVE_DB_PATH

from spaceinvasion import game_data
from spaceinvasion.action_graph import (
    build_action_graph,
    merge_strategy_config_overrides,
)
from spaceinvasion.config import StrategyConfig
from spaceinvasion.fleet import SpyReport
from spaceinvasion.logging_utils import redact_sensitive
from spaceinvasion.optimizer import (
    STATE_FEATURE_DIM,
    TORCH_AVAILABLE,
    GenerationStats,
    NeuralValueNetwork,
    OptimizerRunSummary,
    PurePythonMLP,
    SimAction,
    TorchMLP,
    create_value_network,
    encode_empire_state,
    load_attack_history_stats,
    optimize_plan_actions,
    run_optimizer,
    simulate_action_sequence,
)
from spaceinvasion.ranking import (
    attach_action_etas,
    compute_action_eta,
    estimate_account_points,
    estimate_hourly_economy,
)
from spaceinvasion.state import (
    BuildingOffer,
    EnergyBalance,
    GameState,
    PlanetState,
    ResearchOffer,
    Resources,
    UnitOffer,
)
from spaceinvasion.storage import GameDatabase
from spaceinvasion.strategy import PlannedAction, plan_actions

try:
    import torch  # type: ignore[import-not-found]

    TORCH_VERSION: str | None = str(getattr(torch, "__version__", "installed"))
except ImportError:
    torch = None  # type: ignore[assignment]
    TORCH_VERSION = None

try:
    _headless_mpl = "ipykernel" not in sys.modules and "MPLBACKEND" not in environ
    if _headless_mpl:
        environ.setdefault("MPLBACKEND", "Agg")
    import matplotlib  # type: ignore[import-not-found]

    if _headless_mpl:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt  # type: ignore[import-not-found]

    HAS_MATPLOTLIB = True
except Exception:
    matplotlib = None  # type: ignore[assignment]
    plt = None  # type: ignore[assignment]
    HAS_MATPLOTLIB = False


def render_banner(title: str) -> str:
    rule = "=" * max(48, len(title) + 6)
    return f"\n{rule}\n  {title}\n{rule}"


def render_table(headers: list[str], rows: list[list[Any]]) -> str:
    """Format rows as an aligned ASCII grid with compact-scientific huge-int safety."""
    str_rows = [
        [
            game_data.fmt_compact_int(cell)
            if isinstance(cell, int) and not isinstance(cell, bool)
            else str(cell)
            for cell in row
        ]
        for row in rows
    ]
    widths = [len(h) for h in headers]
    for row in str_rows:
        for idx, cell in enumerate(row):
            if idx < len(widths):
                widths[idx] = max(widths[idx], len(cell))
    sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"
    head_line = "| " + " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers)) + " |"
    lines = [sep, head_line, sep]
    for row in str_rows:
        padded = [
            (row[i] if i < len(row) else "").ljust(widths[i])
            for i in range(len(widths))
        ]
        lines.append("| " + " | ".join(padded) + " |")
    lines.append(sep)
    return "\n".join(lines)


def ascii_bar(value: float, max_value: float, width: int = 20) -> str:
    """Render a `[####....]` bar guarded against NaN, Inf, booleans, and huge integers."""
    safe_width = max(0, min(10_000, game_data.safe_int(width, 20)))
    if (
        isinstance(value, bool)
        or isinstance(max_value, bool)
        or value is None
        or max_value is None
    ):
        return "[" + ("." * safe_width) + "]"
    ratio = game_data.safe_big_ratio(value, max_value, cap=1.0)
    if not math.isfinite(ratio) or ratio <= 0.0:
        return "[" + ("." * safe_width) + "]"
    ratio = max(0.0, min(1.0, ratio))
    filled = int(round(ratio * safe_width))
    return "[" + ("#" * filled) + ("." * (safe_width - filled)) + "]"


print(render_banner("0. Environment & Runtime Summary"))
print(
    render_table(
        ["Setting", "Value"],
        [
            ["REPO_ROOT", str(REPO_ROOT)],
            ["ACTIVE_DATA_ROOT", str(ACTIVE_DATA_ROOT)],
            ["ATTACK_HISTORY_PATH", f"{ATTACK_HISTORY_PATH} (exists={ATTACK_HISTORY_PATH.is_file()})"],
            ["LIVE_DB_PATH", f"{LIVE_DB_PATH} (exists={LIVE_DB_PATH.is_file()})"],
            ["STATE_FEATURE_DIM", STATE_FEATURE_DIM],
            ["TORCH_AVAILABLE", f"{TORCH_AVAILABLE} ({TORCH_VERSION or 'PurePythonMLP fallback'})"],
            ["HAS_MATPLOTLIB", HAS_MATPLOTLIB],
        ],
    )
)

## 1A. Tracked `attack_history.json` Schema & Safe Scalar Coercion

`data/attack_history.json` stores aggregated raid outcomes and high-level telemetry counters so the `ga_nn` planner can bootstrap historical target win-rate priors even before querying SQLite.

- **Merchant Parity (`SpiceEq`)**: SpaceInvasion Classic values resources at `6 Pig Iron : 2.5 Metal : 3 Kryptonite : 1 Spice`, so loot vectors convert to Spice-equivalent units via:
  $$\text{SpiceEq} = \frac{\text{Pig Iron}}{6} + \frac{\text{Metal}}{2.5} + \frac{\text{Kryptonite}}{3} + \text{Spice}$$
- **Defensive Coercion (`_safe_float`, `_safe_int`)**: Rejects `NaN`, `Inf`, booleans, containers, non-ASCII numeral strings, and `>4300`-digit overflows so malformed or cold-start JSON files degrade gracefully to safe defaults.

In [ ]:
def _as_dict(val: Any) -> dict[str, Any]:
    return dict(val) if isinstance(val, dict) else {}


def _as_list(val: Any) -> list[Any]:
    return list(val) if isinstance(val, list) else []


def _safe_float(val: Any, default: float = 0.0) -> float:
    if isinstance(val, (bool, dict, list, tuple, set, frozenset)) or val is None:
        return default
    if isinstance(val, (bytes, bytearray)):
        val = bytes(val).decode("utf-8", errors="replace")
    if isinstance(val, str):
        txt = val.strip()
        if not txt or not txt.isascii():
            return default
        val = txt
    try:
        out = float(val)
    except (TypeError, ValueError, OverflowError):
        return default
    return out if math.isfinite(out) else default


def _safe_int(val: Any, default: int = 0) -> int:
    if isinstance(val, (bool, dict, list, tuple, set, frozenset)) or val is None:
        return default
    if isinstance(val, (bytes, bytearray)):
        val = bytes(val).decode("utf-8", errors="replace")
    try:
        if isinstance(val, str):
            txt = val.strip()
            if not txt or not txt.isascii():
                return default
            try:
                return int(txt)
            except ValueError:
                fval = float(txt)
                return int(fval) if math.isfinite(fval) else default
        if isinstance(val, float) and not math.isfinite(val):
            return default
        return int(val)
    except (TypeError, ValueError, OverflowError):
        return default


disk_attack_history = _as_dict(load_attack_history_stats(ATTACK_HISTORY_PATH))
attack_history_sha256: str | None = None
try:
    if ATTACK_HISTORY_PATH.is_file():
        attack_history_sha256 = hashlib.sha256(ATTACK_HISTORY_PATH.read_bytes()).hexdigest()
except OSError:
    attack_history_sha256 = None

disk_overall = _as_dict(disk_attack_history.get("overall"))
disk_per_target = {
    game_data.safe_key(k): _as_dict(v)
    for k, v in _as_dict(disk_attack_history.get("per_target")).items()
}
disk_history_rows = [
    _as_dict(row)
    for row in _as_list(disk_attack_history.get("history"))
    if isinstance(row, dict)
]
disk_telemetry = _as_dict(disk_attack_history.get("telemetry_summary"))
disk_tel_planet_snaps = _as_dict(disk_telemetry.get("planet_snapshots"))
disk_tel_rank_snaps = _as_dict(disk_telemetry.get("rank_snapshots"))
disk_tel_action_hist = _as_dict(disk_telemetry.get("action_history"))
disk_tel_spy_targets = _as_dict(disk_telemetry.get("spy_targets"))
disk_tel_raid_hist = _as_dict(disk_telemetry.get("raid_history"))
disk_last_atk = _as_dict(disk_overall.get("last_attack"))
disk_last_loot_sp = _safe_float(disk_last_atk.get("loot_spice_eq"), 0.0)
disk_action_cnt = _safe_int(
    disk_telemetry.get("action_count"),
    _safe_int(disk_tel_action_hist.get("total"), 0),
)
disk_planet_snap_cnt = _safe_int(
    disk_telemetry.get("planet_snapshots_count"),
    _safe_int(disk_tel_planet_snaps.get("count"), 0),
)
disk_rank_snap_cnt = _safe_int(
    disk_telemetry.get("rank_snapshots_count"),
    _safe_int(disk_tel_rank_snaps.get("count"), 0),
)
disk_spy_cnt = _safe_int(
    disk_telemetry.get("spy_targets_count"),
    _safe_int(disk_tel_spy_targets.get("count"), 0),
)
disk_raid_tel_cnt = _safe_int(
    disk_tel_raid_hist.get("count"),
    _safe_int(disk_tel_raid_hist.get("attempts"), len(disk_history_rows)),
)

print(render_banner("1A. Read-Only Tracked Attack History JSON Summary"))
print(
    render_table(
        ["Metric", "Value"],
        [
            ["SHA-256", attack_history_sha256 or "N/A (file absent)"],
            ["Attempts / Wins / Losses", f"{_safe_int(disk_overall.get('attempts'))} / {_safe_int(disk_overall.get('wins'))} / {_safe_int(disk_overall.get('losses'))}"],
            ["Success Rate", f"{_safe_float(disk_overall.get('success_rate')):.2%}"],
            ["Total Loot (SpiceEq)", f"{_safe_float(disk_overall.get('total_loot_spice_eq')):,.1f}"],
            ["Avg / Recency SpiceEq", f"{_safe_float(disk_overall.get('avg_loot_spice_eq')):,.1f} / {_safe_float(disk_overall.get('recency_weighted_loot_spice_eq')):,.1f}"],
            ["Last Attack Loot (SpiceEq)", f"{disk_last_loot_sp:,.1f}"],
            ["Tracked Targets / History Rows", f"{len(disk_per_target)} / {len(disk_history_rows)} (tel_raids={disk_raid_tel_cnt}, spy={disk_spy_cnt})"],
            ["Telemetry Actions / Snapshots", f"{disk_action_cnt} actions, {disk_planet_snap_cnt} planet snaps, {disk_rank_snap_cnt} rank snaps"],
        ],
    )
)

## 1B. Zero-Mutation SQLite WAL Inspection & MER Relational Rollup (`state/spaceinvasion.db`)

Aligned with [documentation/database-mer.md](../documentation/database-mer.md), `inspect_sqlite_readonly()` copies `spaceinvasion.db` (+ `-wal`, `-shm`, `-journal` sidecars) into a temporary directory, validates all **20 MER tables** and **16 secondary indexes** via a **4-layer preflight**, and executes cross-domain relational SQL rollups:

1. **Physical Page & B-Tree Integrity**: Verifies `file_size % page_size == 0` and full `PRAGMA integrity_check;`.
2. **20-Table & 16-Index MER Schema Verification**: Validates table types, columns, and index bindings before instantiating `GameDatabase`.
3. **Savepoint Migration Probe**: Validates legacy `spy_targets` column migrations and `meta` savepoint writes cleanly on the temp copy.
4. **MER Relational Queries & Scrubbing**: Joins `BATTLE_REPORTS` $\bowtie$ `GAME_MESSAGES` $\bowtie$ `SPY_TARGETS` (`is_own_planet = 0`, `merged_into = ''`), `RECYCLE_REPORTS`, `BASH_BLOCKS`, and `MANUAL_QUEUE` $\bowtie$ `MANUAL_QUEUE_EVENTS` with `redact_sensitive` scrubbing.

In [ ]:
_EXPECTED_DB_TABLES: dict[str, set[str]] = {
    # Domain 1: Empire Telemetry & Configuration
    "meta": {"key", "value", "updated_ts"},
    "config_history": {"id", "ts", "config_json"},
    "rank_snapshots": {
        "id", "ts", "server_points", "server_rank", "estimated_points",
        "pts_per_hr", "spice_value", "planets",
    },
    "planet_snapshots": {
        "id", "ts", "planet_id", "coords", "planet_type", "fields_used",
        "fields_max", "pig_iron", "metal", "kryptonite", "spice",
        "energy_produced", "energy_used", "buildings_json", "ships_json",
        "economy_rates_json",
    },
    "economy_rates_cache": {"planet_id", "coords", "rates_json", "updated_ts"},
    "tutorial_progress": {
        "step", "total_steps", "tasks_json", "reward_text", "can_complete",
        "first_seen_ts", "last_seen_ts", "completed_ts",
    },
    # Domain 2: Action Execution & Operator Manual Queue
    "action_history": {
        "id", "ts", "source", "command", "priority", "kind", "planet_coords",
        "target_name", "target_level", "executed", "dry_run", "message",
    },
    "manual_queue": {"id", "created_ts", "updated_ts", "status", "priority", "kind"},
    "manual_queue_events": {"id", "ts", "queue_id", "event_type"},
    # Domain 3: Galaxy Mapping & Colonization Intelligence
    "galaxy_scans": {"galaxy", "system", "last_scanned_ts"},
    "galaxy_scan_history": {"id", "ts", "galaxy", "system"},
    "galaxy_slots": {"coords", "galaxy", "system", "position", "last_scanned_ts"},
    "galaxy_slot_history": {"id", "ts", "coords", "galaxy", "system", "position"},
    # Domain 4: Espionage & Target Registry
    "spy_targets": {
        "coords", "owner", "is_inactive", "is_spacia", "points", "pig_iron",
        "metal", "kryptonite", "spice", "fleet_json", "defense_json", "dug_in",
        "last_scanned_ts",
    },
    "spy_reports": {"id", "ts", "target_coords", "pig_iron", "metal", "kryptonite", "spice"},
    # Domain 5: In-Game Inbox, Combat, Bash Regulation & Recycling
    "game_messages": {"id", "ingested_ts", "msg_uid", "category"},
    "raid_history": {
        "id", "target_coords", "attacked_ts", "is_exempt", "outcome", "loot_json",
    },
    "bash_blocks": {"id", "target_coords", "blocked_ts", "blocked_until_ts"},
    "battle_reports": {
        "id", "ts", "msg_uid", "target_coords", "outcome",
        "pig_iron", "metal", "kryptonite", "spice", "loot_spice_eq",
    },
    "recycle_reports": {
        "id", "ts", "msg_uid", "coords", "harvested_spice_eq",
    },
}
_EXPECTED_DB_INDEXES: dict[str, str] = {
    "idx_rank_snapshots_ts": "rank_snapshots",
    "idx_planet_snapshots_pid_ts": "planet_snapshots",
    "idx_action_history_ts": "action_history",
    "idx_spy_reports_coords_ts": "spy_reports",
    "idx_game_messages_ts": "game_messages",
    "idx_game_messages_coords": "game_messages",
    "idx_game_messages_cat": "game_messages",
    "idx_galaxy_scans_ts": "galaxy_scans",
    "idx_galaxy_scan_history_sys_ts": "galaxy_scan_history",
    "idx_galaxy_slots_sys": "galaxy_slots",
    "idx_galaxy_slot_history_coords_ts": "galaxy_slot_history",
    "idx_config_history_ts": "config_history",
    "idx_raid_history_coords_ts": "raid_history",
    "idx_bash_blocks_coords_ts": "bash_blocks",
    "idx_battle_reports_coords_ts": "battle_reports",
    "idx_recycle_reports_coords_ts": "recycle_reports",
}
_SPY_MIGRATED_COLS: tuple[str, ...] = ("planet_name", "status_color", "last_spied_ts")


def inspect_sqlite_readonly(db_path: Path) -> dict[str, Any]:
    """Copy DB (+ WAL/SHM/journal) into a temp dir, run 20-table MER preflight, and query relationally."""
    out: dict[str, Any] = {
        "exists": False,
        "size_bytes": 0,
        "counts": {},
        "raid_stats": {
            "overall": {"attempts": 0, "wins": 0, "losses": 0, "success_rate": 0.0},
            "per_target": {},
            "history": [],
            "telemetry_summary": {},
        },
        "full_history_summary": {},
        "rank_samples_tail": [],
        "recent_actions_sanitized": [],
        "mer_rollup": {},
        "error": None,
    }
    try:
        exists = db_path.is_file()
        out["exists"] = exists
        if not exists:
            return out
        size_bytes = db_path.stat().st_size
        out["size_bytes"] = size_bytes
        if size_bytes == 0:
            return out
        with tempfile.TemporaryDirectory(prefix="si_nb_ro_") as tmp_dir:
            tmp_root = Path(tmp_dir)
            tmp_db_path = tmp_root / db_path.name
            shutil.copy2(db_path, tmp_db_path)
            tmp_db_path.chmod(0o600)
            for suffix in ("-wal", "-shm", "-journal"):
                sidecar = Path(f"{db_path}{suffix}")
                if sidecar.is_file():
                    tmp_sidecar = Path(f"{tmp_db_path}{suffix}")
                    shutil.copy2(sidecar, tmp_sidecar)
                    tmp_sidecar.chmod(0o600)

            tmp_db_size = tmp_db_path.stat().st_size
            if tmp_db_size < 100:
                raise sqlite3.DatabaseError("file is not a database")

            with contextlib.closing(sqlite3.connect(tmp_db_path)) as probe_conn:
                # Layer 1: Physical alignment & full multi-page B-tree/index/constraint integrity.
                page_size_row = probe_conn.execute("PRAGMA page_size;").fetchone()
                page_size = int(page_size_row[0]) if page_size_row else 0
                if page_size <= 0 or tmp_db_size % page_size != 0:
                    raise sqlite3.DatabaseError("truncated or misaligned SQLite database file")
                ic_rows = probe_conn.execute("PRAGMA integrity_check;").fetchall()
                if [tuple(r) for r in ic_rows] != [("ok",)]:
                    msg = ic_rows[0][0] if ic_rows and ic_rows[0] else "integrity check failed"
                    raise sqlite3.DatabaseError(f"SQLite integrity_check failed: {msg}")

                # Layer 2: Case-insensitive sqlite_master schema, index, column & casing validation.
                schema_rows = probe_conn.execute(
                    "SELECT name, type, tbl_name, sql FROM sqlite_master WHERE type != 'trigger'"
                ).fetchall()
                objects_by_name = {
                    str(name).lower(): (
                        str(name),
                        str(obj_type).lower(),
                        str(tbl_name).lower(),
                        str(sql or ""),
                    )
                    for name, obj_type, tbl_name, sql in schema_rows
                }
                table_cols_map: dict[str, set[str]] = {}
                for tbl_name, req_cols in _EXPECTED_DB_TABLES.items():
                    if tbl_name not in objects_by_name:
                        continue
                    orig_name, obj_type, _, obj_sql = objects_by_name[tbl_name]
                    if obj_type != "table" or obj_sql.lstrip().upper().startswith(
                        "CREATE VIRTUAL TABLE"
                    ):
                        raise sqlite3.DatabaseError(
                            f"Incompatible SQLite schema object {tbl_name!r}: type={obj_type!r}"
                        )
                    safe_ident = orig_name.replace('"', '""')
                    cols_exact = {
                        str(row[1])
                        for row in probe_conn.execute(
                            f'PRAGMA table_info("{safe_ident}");'
                        ).fetchall()
                    }
                    cols_lower = {c.lower() for c in cols_exact}
                    table_cols_map[tbl_name] = cols_lower
                    missing_cols = sorted(req_cols - cols_lower)
                    if missing_cols:
                        raise sqlite3.DatabaseError(
                            f"Incompatible SQLite table {tbl_name!r}: missing columns {missing_cols}"
                        )
                    if tbl_name == "spy_targets":
                        for mcol in _SPY_MIGRATED_COLS:
                            if mcol in cols_lower and mcol not in cols_exact:
                                raise sqlite3.DatabaseError(
                                    f"Incompatible SQLite column casing for {mcol!r} in spy_targets"
                                )
                for idx_name, expected_tbl in _EXPECTED_DB_INDEXES.items():
                    if idx_name not in objects_by_name:
                        continue
                    _, obj_type, tbl_name, _ = objects_by_name[idx_name]
                    if obj_type != "index" or tbl_name != expected_tbl:
                        raise sqlite3.DatabaseError(
                            f"Incompatible SQLite index object {idx_name!r}: "
                            f"type={obj_type!r}, tbl={tbl_name!r}"
                        )

                # Layer 3: Write/DDL simulation matching GameDatabase.__init__.
                probe_conn.execute("PRAGMA journal_mode=WAL;").fetchall()
                probe_conn.execute("PRAGMA synchronous=NORMAL;")
                probe_conn.execute("PRAGMA foreign_keys=ON;")
                if "spy_targets" in objects_by_name:
                    spy_ident = objects_by_name["spy_targets"][0].replace('"', '""')
                    spy_cols_exact = {
                        str(row[1])
                        for row in probe_conn.execute(
                            f'PRAGMA table_info("{spy_ident}");'
                        ).fetchall()
                    }
                    if "planet_name" not in spy_cols_exact:
                        probe_conn.execute(
                            f'ALTER TABLE "{spy_ident}" ADD COLUMN planet_name TEXT NOT NULL DEFAULT \'\''
                        )
                    if "status_color" not in spy_cols_exact:
                        probe_conn.execute(
                            f'ALTER TABLE "{spy_ident}" ADD COLUMN status_color TEXT NOT NULL DEFAULT \'yellow\''
                        )
                    if "last_spied_ts" not in spy_cols_exact:
                        probe_conn.execute(
                            f'ALTER TABLE "{spy_ident}" ADD COLUMN last_spied_ts REAL NOT NULL DEFAULT 0.0'
                        )
                    if probe_conn.in_transaction:
                        probe_conn.commit()
                if "meta" in objects_by_name:
                    meta_ident = objects_by_name["meta"][0].replace('"', '""')
                    probe_conn.execute("SAVEPOINT _si_preflight_meta;")
                    probe_conn.execute(
                        f'INSERT OR IGNORE INTO "{meta_ident}"(key, value, updated_ts) VALUES(?, ?, ?)',
                        ("schema_version", "1", time.time()),
                    )
                    probe_conn.execute("ROLLBACK TO SAVEPOINT _si_preflight_meta;")
                    probe_conn.execute("RELEASE SAVEPOINT _si_preflight_meta;")
                if probe_conn.in_transaction:
                    probe_conn.rollback()

            # Layer 4: Construct GameDatabase + execute MER cross-domain relational rollup queries.
            ro_db = GameDatabase(
                tmp_db_path,
                attack_history_path=tmp_root / "ro_mirror.json",
            )
            try:
                db_sum = _as_dict(ro_db.summary())
                counts_val = _as_dict(db_sum.get("counts"))
                raid_stats_val = redact_sensitive(ro_db.get_raid_stats())
                full_summary_val = redact_sensitive(ro_db.get_full_history_summary())
                rank_tail_val = redact_sensitive(ro_db.get_rank_samples(limit=5))

                def _decode_text(val: Any) -> Any:
                    if isinstance(val, (bytes, bytearray)):
                        return val.decode("utf-8", errors="replace")
                    return val

                clean_actions: list[dict[str, Any]] = []
                for row in ro_db.get_recent_actions(limit=5):
                    row_d = _as_dict(row)
                    clean_actions.append(
                        {
                            "ts": row_d.get("ts"),
                            "kind": redact_sensitive(_decode_text(row_d.get("kind"))),
                            "priority": redact_sensitive(_decode_text(row_d.get("priority"))),
                            "planet_coords": redact_sensitive(_decode_text(row_d.get("planet_coords"))),
                            "target_name": redact_sensitive(_decode_text(row_d.get("target_name"))),
                            "executed": bool(row_d.get("executed")),
                            "dry_run": bool(row_d.get("dry_run")),
                        }
                    )

                # MER Relational Rollup across BATTLE_REPORTS, GAME_MESSAGES, SPY_TARGETS, RECYCLE_REPORTS & MANUAL_QUEUE
                mer_rollup: dict[str, Any] = {
                    "battle_wins": 0,
                    "battle_losses": 0,
                    "battle_gross_spice_eq": 0.0,
                    "battle_net_spice_eq": 0.0,
                    "recycle_total_spice_eq": 0.0,
                    "active_farm_targets": 0,
                    "colony_300plus_candidates": 0,
                    "pending_manual_queue": 0,
                }
                with contextlib.closing(sqlite3.connect(tmp_db_path)) as q_conn:
                    q_tables = {
                        str(r[0]).lower()
                        for r in q_conn.execute(
                            "SELECT name FROM sqlite_master WHERE type='table'"
                        ).fetchall()
                    }
                    if {"battle_reports", "game_messages", "spy_targets"}.issubset(q_tables):
                        br_row = q_conn.execute(
                            """
                            SELECT
                                SUM(CASE WHEN LOWER(br.outcome) IN ('win', 'victory') THEN 1 ELSE 0 END),
                                SUM(CASE WHEN LOWER(br.outcome) IN ('win', 'victory') THEN 0 ELSE 1 END),
                                COALESCE(SUM(br.loot_spice_eq), 0.0),
                                COALESCE(SUM(br.loot_spice_eq - br.attacker_lost_spice), 0.0)
                            FROM battle_reports br
                            LEFT JOIN game_messages gm ON gm.msg_uid = br.msg_uid
                            LEFT JOIN spy_targets st
                                   ON st.coords = br.target_coords
                                  AND COALESCE(st.is_own_planet, 0) = 0
                                  AND COALESCE(st.merged_into, '') = ''
                            """
                        ).fetchone()
                        if br_row:
                            mer_rollup["battle_wins"] = _safe_int(br_row[0], 0)
                            mer_rollup["battle_losses"] = _safe_int(br_row[1], 0)
                            mer_rollup["battle_gross_spice_eq"] = _safe_float(br_row[2], 0.0)
                            mer_rollup["battle_net_spice_eq"] = _safe_float(br_row[3], 0.0)
                    if "recycle_reports" in q_tables:
                        rec_sum = q_conn.execute(
                            "SELECT COALESCE(SUM(harvested_spice_eq), 0.0) FROM recycle_reports"
                        ).fetchone()
                        if rec_sum:
                            mer_rollup["recycle_total_spice_eq"] = _safe_float(rec_sum[0], 0.0)
                    if "spy_targets" in q_tables:
                        st_cols_now = {
                            str(r[1]).lower()
                            for r in q_conn.execute("PRAGMA table_info('spy_targets')").fetchall()
                        }
                        if {"is_own_planet", "merged_into", "uninhabited_fields"}.issubset(st_cols_now):
                            st_row = q_conn.execute(
                                """
                                SELECT
                                    SUM(CASE WHEN (is_inactive = 1 OR is_spacia = 1) THEN 1 ELSE 0 END),
                                    SUM(CASE WHEN COALESCE(uninhabited_fields, 0) >= 300 THEN 1 ELSE 0 END)
                                FROM spy_targets
                                WHERE COALESCE(is_own_planet, 0) = 0
                                  AND COALESCE(merged_into, '') = ''
                                """
                            ).fetchone()
                            if st_row:
                                mer_rollup["active_farm_targets"] = _safe_int(st_row[0], 0)
                                mer_rollup["colony_300plus_candidates"] = _safe_int(st_row[1], 0)
                    if {"manual_queue", "manual_queue_events"}.issubset(q_tables):
                        mq_row = q_conn.execute(
                            """
                            SELECT COUNT(DISTINCT mq.id)
                            FROM manual_queue mq
                            LEFT JOIN manual_queue_events mqe ON mqe.queue_id = mq.id
                            WHERE mq.status = 'pending'
                            """
                        ).fetchone()
                        if mq_row:
                            mer_rollup["pending_manual_queue"] = _safe_int(mq_row[0], 0)

                out["counts"] = counts_val
                out["raid_stats"] = raid_stats_val
                out["full_history_summary"] = full_summary_val
                out["rank_samples_tail"] = rank_tail_val
                out["recent_actions_sanitized"] = clean_actions
                out["mer_rollup"] = mer_rollup
            finally:
                ro_db.close()
    except (
        sqlite3.Error,
        OSError,
        ValueError,
        TypeError,
        OverflowError,
        KeyError,
        AttributeError,
    ) as exc:
        out["error"] = redact_sensitive(str(exc)) or type(exc).__name__
    return out


live_db_inspection = inspect_sqlite_readonly(LIVE_DB_PATH)
live_counts = _as_dict(live_db_inspection.get("counts"))
live_full_sum = _as_dict(live_db_inspection.get("full_history_summary"))
live_mer = _as_dict(live_db_inspection.get("mer_rollup"))

print(render_banner("1B. Read-Only 20-Table MER SQLite Telemetry & Relational Rollup"))
print(
    render_table(
        ["MER Domain / Metric", "Value"],
        [
            ["DB File Present / Size", f"{live_db_inspection['exists']} / {live_db_inspection['size_bytes']:,} bytes"],
            ["Read Error", live_db_inspection["error"] or "None"],
            ["raid_history / spy_targets (active farms)", f"{_safe_int(live_counts.get('raid_history'))} / {_safe_int(live_counts.get('spy_targets'))} ({_safe_int(live_mer.get('active_farm_targets'))} active farms)"],
            ["planet_snapshots / rank_snapshots", f"{_safe_int(live_counts.get('planet_snapshots'))} / {_safe_int(live_counts.get('rank_snapshots'))}"],
            ["action_history / manual_queue (pending)", f"{_safe_int(live_counts.get('action_history'))} / {_safe_int(live_counts.get('manual_queue'))} ({_safe_int(live_mer.get('pending_manual_queue'))} pending)"],
            ["galaxy_scans / 300+ field colony slots", f"{_safe_int(live_counts.get('galaxy_scans'))} / {_safe_int(live_mer.get('colony_300plus_candidates'))} slots >=300f"],
            ["game_messages / spy_reports", f"{_safe_int(live_counts.get('game_messages'))} / {_safe_int(live_counts.get('spy_reports'))}"],
            ["battle_reports (W/L, Gross -> Net SpEq)", f"{_safe_int(live_mer.get('battle_wins'))}W-{_safe_int(live_mer.get('battle_losses'))}L ({_safe_float(live_mer.get('battle_gross_spice_eq')):,.1f} -> {_safe_float(live_mer.get('battle_net_spice_eq')):+,.1f} SpEq)"],
            ["recycle_reports (Total Harvested SpEq)", f"{_safe_int(live_counts.get('recycle_reports'))} ({_safe_float(live_mer.get('recycle_total_spice_eq')):,.1f} SpEq)"],
            ["Points Delta / Pts-per-hr", f"{_safe_float(live_full_sum.get('points_delta')):.2f} / {_safe_float(live_full_sum.get('pts_per_hr')):.2f}"],
        ],
    )
)

## 2A. Reference Midgame Empire Builder & SpaceInvasion Classic Rules

To test and compare planners reproducibly without touching any live account, `build_sample_game_state()` constructs a SpaceInvasion Classic midgame colony (`Homeworld [02:380:12]`, `5x` universe speed) obeying exact wiki mechanics:

- **Prerequisite-Legal Offers**: Uses `game_data.missing_prerequisites(...)` across buildings, ships, and researches so locked tech/units (`can_research=False` / `can_build=False`, `status_text="Locked"`) are never offered as illegal immediate actions.
- **Energy Technology Scaling**: Applies `r2` Energy technology's `+1%` per level multiplier to positive power output (`Fusion plant` `b5`), while `Blast furnace` (`b2`) consumes `2:1` Pig iron without `r1` Mining technology boost.
- **Anti-Bias Target Ordering**: Places the lower historical win-rate target (`02:380:05 TargetBeta`) **first** in `raid_targets` so naive index-0 selection picks the wrong target unless the optimizer consults raid history.

In [ ]:
def _res_from_cost(raw_cost: dict[str, Any]) -> Resources:
    return Resources(
        pig_iron=int(raw_cost.get("pig_iron", 0) or 0),
        metal=int(raw_cost.get("metal", 0) or 0),
        kryptonite=int(raw_cost.get("kryptonite", 0) or 0),
        spice=int(raw_cost.get("spice", 0) or 0),
    )


def build_sample_game_state(
    *,
    timestamp: float = 1_700_000_000.0,
    resources: Resources = Resources(9_000, 7_000, 4_500, 2_200),
    buildings: dict[str, int] | None = None,
    ships: dict[str, int] | None = None,
    researches: dict[str, int] | None = None,
    include_raid_targets: bool = True,
) -> GameState:
    """Build a prerequisite-legal midgame `GameState` for deterministic offline experiments."""
    bld = dict(
        buildings
        if buildings is not None
        else {
            "b1": 7,
            "b2": 5,
            "b3": 6,
            "b4": 4,
            "b5": 10,
            "b7": 3,
            "b8": 2,
            "b9": 3,
            "b11": 3,
            "b12": 3,
            "b13": 3,
            "b14": 3,
        }
    )
    res_tech = dict(
        researches
        if researches is not None
        else {"r1": 2, "r2": 2, "r4": 2, "r7": 2, "r8": 1}
    )
    shp = dict(
        ships
        if ships is not None
        else {"s1": 12, "s4": 28, "s6": 4, "s9": 4}
    )

    e_prod = 0
    e_used = 0
    for bkey, lvl in bld.items():
        if bkey.startswith("b") and bkey[1:].isascii() and bkey[1:].isdigit() and lvl > 0:
            eval_e = int(
                game_data.building_level_stats(int(bkey[1:]), int(lvl)).get("energy", 0)
                or 0
            )
            if eval_e >= 0:
                e_prod += eval_e
            else:
                e_used += -eval_e
    e_prod_eff = int(round(e_prod * (1.0 + 0.01 * int(res_tech.get("r2", 0)))))

    candidate_bids = sorted(
        {
            1, 2, 3, 4, 5, 7, 8, 9, 11, 12, 13, 14,
            *(
                int(k[1:])
                for k, v in bld.items()
                if k.startswith("b") and k[1:].isascii() and k[1:].isdigit() and v > 0
            ),
        }
    )
    b_offers: list[BuildingOffer] = []
    for bid in candidate_bids:
        info = game_data.BUILDING_BY_ID.get(bid) or {}
        cur = int(bld.get(f"b{bid}", 0))
        nxt = cur + 1
        cost = _res_from_cost(game_data.building_cost(bid, nxt))
        unmet = game_data.missing_prerequisites(
            "buildings", bid, buildings=bld, researches=res_tech
        )
        can_b = (not unmet) and resources.can_afford(cost)
        status_b = (
            f"Build level {nxt}"
            if can_b
            else ("Locked" if unmet else "Missing resources")
        )
        b_offers.append(
            BuildingOffer(
                building_id=bid,
                name=str(info.get("name", f"Building {bid}")),
                current_level=cur,
                target_level=nxt,
                cost=cost,
                can_build=can_b,
                status_text=status_b,
                build_href=f"indexInternal.es?action=internalBuildItem&buildingID={bid}",
            )
        )

    ship_specs = (
        (1, "small-transport"),
        (4, "small-fighter"),
        (6, "starcruiser"),
        (9, "solar-cells"),
    )
    s_offers: list[UnitOffer] = []
    for uid, slug in ship_specs:
        uinfo = game_data.SHIPS.get(slug) or {}
        cost = _res_from_cost(uinfo.get("base_cost") or {})
        unmet_s = game_data.missing_prerequisites(
            "ships", slug, buildings=bld, researches=res_tech
        )
        can_s = (not unmet_s) and resources.can_afford(cost)
        status_s = (
            "Buildable"
            if can_s
            else ("Locked" if unmet_s else "Missing resources")
        )
        s_offers.append(
            UnitOffer(
                unit_type="ship",
                unit_id=uid,
                name=str(uinfo.get("name", slug)),
                owned=int(shp.get(f"s{uid}", 0)),
                cost=cost,
                can_build=can_s,
                input_name=f"ship[{uid}]",
                status_text=status_s,
            )
        )

    r_offers: list[ResearchOffer] = []
    for rid, rkey in ((1, "r1"), (2, "r2"), (4, "r4"), (8, "r8")):
        rinfo = game_data.RESEARCH_BY_ID.get(rkey) or {}
        cur = int(res_tech.get(rkey, 0))
        nxt = cur + 1
        lvl_data = (rinfo.get("levels") or {}).get(nxt) or {}
        cost = _res_from_cost(lvl_data.get("cost") or {})
        unmet_r = game_data.missing_prerequisites(
            "research", rkey, buildings=bld, researches=res_tech
        )
        can_r = (not unmet_r) and (not cost.is_zero()) and resources.can_afford(cost)
        status_r = (
            "Ready"
            if can_r
            else ("Locked" if unmet_r else "Missing resources")
        )
        r_offers.append(
            ResearchOffer(
                research_id=rid,
                research_key=rkey,
                name=str(rinfo.get("name", rkey)),
                current_level=cur,
                target_level=nxt,
                cost=cost,
                can_research=can_r,
                status_text=status_r,
                research_href=f"indexInternal.es?action=internalBuildItem&researchID={rid}",
            )
        )

    planet = PlanetState(
        planet_id=12476,
        name="Homeworld",
        coords="02:380:12",
        planet_type="normal",
        temperature=31,
        fields_used=sum(bld.values()),
        fields_max=230,
        resources=resources,
        storage=Resources(100_000, 100_000, 100_000, 100_000),
        energy=EnergyBalance(produced=e_prod_eff, used=e_used),
        buildings=bld,
        building_names={
            f"b{bid}": str((game_data.BUILDING_BY_ID.get(bid) or {}).get("name", f"b{bid}"))
            for bid in range(1, 15)
            if f"b{bid}" in bld
        },
        ships=shp,
        defense={},
        building_offers=tuple(b_offers),
        ship_offers=tuple(s_offers),
        economy_rates={
            "resource1": 100,
            "resource2": 100,
            "resource3": 100,
            "resource4": 100,
            "resource5": 100,
        },
    )

    raid_targets: tuple[dict[str, Any], ...] = ()
    if include_raid_targets:
        # Place lower historical win-rate target first so ordering cannot bias optimizer results.
        raid_targets = (
            {
                "coords": "02:380:05",
                "owner": "TargetBeta",
                "planet_name": "InactiveOutpostB",
                "status_color": "yellow",
                "is_inactive": True,
                "is_spacia": False,
                "resources": {
                    "pig_iron": 22_000,
                    "metal": 18_000,
                    "kryptonite": 10_000,
                    "spice": 5_000,
                },
                "fleet": {},
                "defense": {},
            },
            {
                "coords": "02:380:04",
                "owner": "TargetAlpha",
                "planet_name": "InactiveFarmA",
                "status_color": "yellow",
                "is_inactive": True,
                "is_spacia": False,
                "resources": {
                    "pig_iron": 22_000,
                    "metal": 18_000,
                    "kryptonite": 10_000,
                    "spice": 5_000,
                },
                "fleet": {},
                "defense": {},
            },
            {
                "coords": "02:380:09",
                "owner": "Spacia",
                "planet_name": "SpaciaSector",
                "status_color": "yellow",
                "is_inactive": True,
                "is_spacia": True,
                "resources": {
                    "pig_iron": 16_000,
                    "metal": 12_000,
                    "kryptonite": 8_000,
                    "spice": 4_000,
                },
                "fleet": {},
                "defense": {"d1": 2},
            },
        )

    return GameState(
        universe_id=31,
        user_id=1001,
        nickname="CommanderDemo",
        timestamp=float(timestamp),
        planets=(planet,),
        active_planet_id=12476,
        researches=res_tech,
        research_names={
            rk: str((game_data.RESEARCH_BY_ID.get(rk) or {}).get("name", rk))
            for rk in res_tech
        },
        specs={"miner": 1, "fleeter": 1},
        research_offers=tuple(r_offers),
        speedup_factor=5.0,
        newbie_protection=False,
        account_points=320,
        account_rank=185,
        raid_targets=raid_targets,
    )


preview_state = build_sample_game_state()
preview_p0 = preview_state.planets[0]
preview_econ = estimate_hourly_economy(preview_state)
preview_pts = estimate_account_points(preview_state)

print(render_banner("2A. Reference Midgame Empire Snapshot"))
print(
    render_table(
        ["Empire Attribute", "Details"],
        [
            ["Colony / Coords / Temp", f"{preview_p0.name} [{preview_p0.coords}] ({preview_p0.temperature}C, {preview_p0.fields_used}/{preview_p0.fields_max} fields)"],
            ["Stockpile (Pig / Met / Kry / Spi)", f"{preview_p0.resources.pig_iron:,} / {preview_p0.resources.metal:,} / {preview_p0.resources.kryptonite:,} / {preview_p0.resources.spice:,}"],
            ["Energy Balance", f"{preview_p0.energy.produced} prod - {preview_p0.energy.used} used = {preview_p0.energy.net:+d} net"],
            [
                "Hourly Net Rates (5x Speed)",
                f"Pig {_safe_float(preview_econ.get('pig_iron_per_hr')):,.0f}/h | "
                f"Met {_safe_float(preview_econ.get('metal_per_hr')):,.0f}/h | "
                f"Kry {_safe_float(preview_econ.get('kryptonite_per_hr')):,.0f}/h | "
                f"Spi {_safe_float(preview_econ.get('spice_per_hr')):,.0f}/h",
            ],
            [
                "Estimated Account Points",
                f"{_safe_float(preview_pts.get('estimated_points')):.2f} pts "
                f"({_safe_float(preview_econ.get('pts_per_hr_steady')):.2f} pts/hr, "
                f"{_safe_float(preview_econ.get('spice_eq_per_hr')):,.1f} SpiceEq/hr)",
            ],
            ["Stationed Fleet", ", ".join(f"{k}:{v}" for k, v in sorted(preview_p0.ships.items()))],
        ],
    )
)

## 2B. Controlled `:memory:` Multi-Table Telemetry Laboratory

Why does **complete attack history** matter more than single-attack memory or surface spy loot?

In our controlled `:memory:` `GameDatabase` seed below, two inactive targets carry **identical spy resources** (`22k Pig / 18k Met / 10k Kry / 5k Spi`) and **identical last-attack loot** (`10k Pig / 5k Met / 2k Kry / 500 Spi`):

| Target | Historical Record | Win Rate | Strategic Ground Truth |
|---|---|---:|---|
| **`02:380:04` (`TargetAlpha`)** | `9` victories, `1` loss | **`90%`** | Reliable **Farm** — high expected plunder |
| **`02:380:05` (`TargetBeta`)** | `1` lucky victory (most recent!), `9` intercepted losses | **`10%`** | **Fluke Trap** — destroys raiders 90% of the time |
| **`02:380:09` (`Spacia`)** | `3` victories, `0` losses (`2x d1` defense) | **`100%`** | **Bash-exempt Spacia** — steady secondary farm |

Seeding `planet_snapshots`, `rank_snapshots`, `spy_targets`, `action_history`, and `raid_history` into `GameDatabase(":memory:")` exercises the full multi-table intelligence pipeline without touching disk.

In [ ]:
base_ts = 1_700_000_000.0
demo_db = GameDatabase(":memory:")
try:
    for step_idx in range(4):
        snap_ts = base_ts - (4 - step_idx) * 3600.0
        snap_bld = {
            "b1": 4 + (step_idx // 2),
            "b2": 3 + (step_idx // 3),
            "b3": 4 + (step_idx // 2),
            "b4": 2 + (step_idx // 3),
            "b5": 8 + (step_idx // 3),
            "b7": 2 + (step_idx // 3),
            "b8": 2,
            "b9": 2 + (step_idx // 3),
            "b11": 2,
            "b12": 2,
            "b13": 2,
            "b14": 2,
        }
        snap_res = Resources(
            8_000 + step_idx * 1_500,
            5_000 + step_idx * 1_000,
            3_500 + step_idx * 650,
            1_800 + step_idx * 350,
        )
        snap_state = build_sample_game_state(
            timestamp=snap_ts,
            resources=snap_res,
            buildings=snap_bld,
            include_raid_targets=False,
        )
        demo_db.record_state_snapshot(
            snap_state,
            planet_min_interval_seconds=0.0,
            rank_min_interval_seconds=0.0,
        )

    demo_db.record_spy_report(
        SpyReport(
            target_coords="02:380:04",
            owner="TargetAlpha",
            resources=Resources(22_000, 18_000, 10_000, 5_000),
            is_spacia=False,
        ),
        is_inactive=True,
        status_color="yellow",
        ts=base_ts - 900.0,
    )
    demo_db.record_spy_report(
        SpyReport(
            target_coords="02:380:05",
            owner="TargetBeta",
            resources=Resources(22_000, 18_000, 10_000, 5_000),
            is_spacia=False,
        ),
        is_inactive=True,
        status_color="yellow",
        ts=base_ts - 840.0,
    )
    demo_db.record_spy_report(
        SpyReport(
            target_coords="02:380:09",
            owner="Spacia",
            resources=Resources(16_000, 12_000, 8_000, 4_000),
            defense={"d1": 2},
            is_spacia=True,
        ),
        is_inactive=True,
        status_color="yellow",
        ts=base_ts - 780.0,
    )

    sample_actions = [
        ("BUILD", "Pig iron mine", 6, True, "Upgraded mine"),
        ("RESEARCH", "Mining technology", 2, True, "Completed research"),
        ("BUILD_SHIP", "Small fighter", 5, True, "Built raiders"),
        ("THROTTLE_BUILDING", "Blast furnace", 100, True, "Restored smelter"),
        ("ATTACK", "02:380:04", None, True, "Raid dispatched"),
        ("BUILD", "Fusion plant", 9, False, "Queue busy"),
    ]
    for idx, (kind, tgt_name, tgt_lvl, ok, msg) in enumerate(sample_actions):
        demo_db.record_action(
            source="experiment",
            command=kind.lower(),
            action=PlannedAction(
                priority="P2",
                kind=kind,
                planet_id=12476,
                planet_coords="02:380:12",
                target_name=tgt_name,
                target_level=tgt_lvl,
                reason="Historical telemetry seed",
                executable_now=True,
            ),
            executed=ok,
            dry_run=False,
            message=msg,
            ts=base_ts - 6000.0 + idx * 600.0,
        )

    # 02:380:04 -> 9/10 wins; 02:380:05 -> 1/10 fluke with identical last-attack loot;
    # 02:380:09 -> 3/3 Spacia wins.
    shared_last_loot = {"pig_iron": 10_000, "metal": 5_000, "kryptonite": 2_000, "spice": 500}
    for i in range(10):
        ts_04 = base_ts - (10 - i) * 43_200.0
        if i == 4:
            demo_db.record_raid(
                "02:380:04",
                attacked_ts=ts_04,
                is_exempt=True,
                outcome="Defeat / bounced",
                loot={"pig_iron": 0, "metal": 0, "kryptonite": 0, "spice": 0},
            )
        else:
            loot_04 = (
                shared_last_loot
                if i == 9
                else {"pig_iron": 12_000, "metal": 7_500, "kryptonite": 3_500, "spice": 1_200}
            )
            demo_db.record_raid(
                "02:380:04",
                attacked_ts=ts_04,
                is_exempt=True,
                outcome="Victory",
                loot=loot_04,
            )

    for i in range(10):
        ts_05 = base_ts - (10 - i) * 43_200.0 + 300.0
        if i < 9:
            demo_db.record_raid(
                "02:380:05",
                attacked_ts=ts_05,
                is_exempt=True,
                outcome="Defeat / intercepted",
                loot={"pig_iron": 0, "metal": 0, "kryptonite": 0, "spice": 0},
            )
        else:
            demo_db.record_raid(
                "02:380:05",
                attacked_ts=ts_05,
                is_exempt=True,
                outcome="Victory",
                loot=shared_last_loot,
            )

    for i in range(3):
        demo_db.record_raid(
            "02:380:09",
            attacked_ts=base_ts - (3 - i) * 86_400.0,
            is_exempt=True,
            outcome="Victory",
            loot={"pig_iron": 8_000, "metal": 6_000, "kryptonite": 3_000, "spice": 1_500},
        )

    demo_stats = demo_db.get_raid_stats()
    demo_full_summary = demo_db.get_full_history_summary()
    demo_db_counts = dict((demo_db.summary() or {}).get("counts") or {})
finally:
    demo_db.close()

demo_overall = demo_stats["overall"]
demo_per_target = demo_stats["per_target"]
max_recency_sp = max(
    (float(pt.get("recency_weighted_loot_spice_eq", 0.0)) for pt in demo_per_target.values()),
    default=1.0,
)
target_rows = []
for coords in sorted(demo_per_target.keys()):
    pt = demo_per_target[coords]
    rec_sp = float(pt["recency_weighted_loot_spice_eq"])
    target_rows.append(
        [
            coords,
            f"{pt['wins']}/{pt['attempts']} ({pt['success_rate']:.0%})",
            f"{pt['total_loot_spice_eq']:,.1f}",
            f"{pt['avg_loot_spice_eq']:,.1f}",
            f"{rec_sp:,.1f}",
            pt.get("last_outcome", ""),
            f"{pt.get('last_loot_spice_eq', 0.0):,.1f}",
            ascii_bar(rec_sp, max_recency_sp, width=12),
        ]
    )

print(render_banner("2B. Isolated :memory: Complete Attack History Audit"))
print(
    f"Overall: {demo_overall['wins']}/{demo_overall['attempts']} wins "
    f"({demo_overall['success_rate']:.1%}), total={demo_overall['total_loot_spice_eq']:,.1f} SpiceEq, "
    f"avg={demo_overall['avg_loot_spice_eq']:,.1f}, recency={demo_overall['recency_weighted_loot_spice_eq']:,.1f}"
)
print(
    render_table(
        ["Target", "Wins/Att (SR)", "Total SpEq", "Avg SpEq", "Recency SpEq", "Last Outcome", "Last SpEq", "Yield Bar"],
        target_rows,
    )
)
print(
    render_table(
        ["Telemetry Signal", "Summary"],
        [
            ["Table Row Counts", json.dumps(demo_db_counts, sort_keys=True, allow_nan=False)],
            ["Rank Points Delta", f"+{demo_full_summary['points_delta']:.2f} pts (latest {demo_full_summary['pts_per_hr']:.2f} pts/hr)"],
            ["Economy Trend", f"+{demo_full_summary['economy_trend']:.2f} pts/hr"],
            ["Action History Success Rate", f"{demo_full_summary['action_success_rate']:.1%} ({demo_full_summary['action_successes']}/{demo_full_summary['action_history_count']})"],
            ["Spy Targets Known SpiceEq", f"{demo_full_summary['spy_targets']['total_known_resources_spice_eq']:,.1f} ({demo_full_summary['spy_targets']['count']} targets)"],
        ],
    )
)

## 3A. 46-Dimensional Empire & Historical State Encoding (`encode_empire_state`)

Every candidate empire state (`GameState` or simulated `_SimState`) is mapped into a bounded, finite `46`-element vector (`STATE_FEATURE_DIM == 46`) divided into four interpretable blocks:

| Index Range | Domain Block | Encoded Signals |
|---|---|---|
| **`[0..15]`** | **Resources, Rates & Energy** | Log-scaled stockpiles, warehouse fill ratios (`0..1`), signed log hourly net production (`Pig`, `Met`, `Kry`, `Spi`, `SpiceEq/h`, `pts/h`), min energy margin & deficit fraction |
| **`[16..26]`** | **Colony Infrastructure** | Normalized building levels (`b1..b9`, warehouses `b11..b14`) and remaining free building-field ratio |
| **`[27..35]`** | **Tech, Fleet & Queues** | Key researches (`r1`, `r2`, engine max, `r7`, combat sum), account points, fleet combat power, cargo capacity & queue occupancy |
| **`[36..45]`** | **Historical Raid & Telemetry Priors** | Overall raid attempts/win-rate/loot, blended average+recency yield, last raid outcome sign, best target win-rate/recency loot, rank trend & action reliability |

In [ ]:
FEATURE_NAMES: tuple[str, ...] = (
    "stock_pig_iron",
    "stock_metal",
    "stock_kryptonite",
    "stock_spice",
    "fill_pig_iron",
    "fill_metal",
    "fill_kryptonite",
    "fill_spice",
    "rate_pig_iron",
    "rate_metal",
    "rate_kryptonite",
    "rate_spice",
    "rate_spice_eq_hr",
    "rate_pts_hr",
    "min_energy_margin",
    "energy_deficit_frac",
    "lvl_b1_pig",
    "lvl_b2_blast",
    "lvl_b3_kryp",
    "lvl_b4_spice",
    "lvl_power_b5_b6",
    "lvl_b7_dev",
    "lvl_b8_lab_max",
    "lvl_b9_wf",
    "lvl_warehouses",
    "free_fields_ratio",
    "tech_r1_mining",
    "tech_r2_energy",
    "tech_engine_max",
    "tech_r7_spy",
    "tech_combat_sum",
    "empire_points",
    "fleet_power",
    "fleet_cargo",
    "queue_occupancy",
    "raid_potential_and_loot",
    "hist_overall_attempts",
    "hist_overall_win_rate",
    "hist_overall_total_loot",
    "hist_overall_blend_avg_rec",
    "hist_last_loot",
    "hist_last_outcome_sign",
    "hist_target_win_rate",
    "hist_target_recency_loot",
    "tel_points_econ_trend",
    "tel_action_sr_and_recon",
)
assert len(FEATURE_NAMES) == STATE_FEATURE_DIM

base_state_no_hist = build_sample_game_state(timestamp=base_ts)
exp_state = replace(base_state_no_hist, raid_stats=demo_stats)
cold_stats = {
    "overall": {"attempts": 0, "wins": 0, "losses": 0, "success_rate": 0.0},
    "per_target": {},
    "history": [],
}

vec_cold = encode_empire_state(base_state_no_hist, raid_stats=cold_stats)
vec_full = encode_empire_state(exp_state, raid_stats=demo_stats)
assert len(vec_cold) == STATE_FEATURE_DIM and len(vec_full) == STATE_FEATURE_DIM
assert all(math.isfinite(x) for x in vec_cold + vec_full)

empire_highlight_indices = (0, 12, 13, 14, 16, 20, 25, 27, 32, 33)
empire_rows = [
    [
        f"[{idx:02d}] {FEATURE_NAMES[idx]}",
        f"{vec_full[idx]:+.4f}",
        ascii_bar(max(0.0, vec_full[idx]), 1.0, width=14),
    ]
    for idx in empire_highlight_indices
]

feat_cmp_rows = []
for idx in range(35, STATE_FEATURE_DIM):
    delta = vec_full[idx] - vec_cold[idx]
    feat_cmp_rows.append(
        [
            f"[{idx:02d}] {FEATURE_NAMES[idx]}",
            f"{vec_cold[idx]:+.4f}",
            f"{vec_full[idx]:+.4f}",
            f"{delta:+.4f}",
            ascii_bar(abs(delta), 1.0, width=12),
        ]
    )

print(render_banner("3A. 46D Empire Feature Highlights & Historical Deltas"))
print(render_table(["Empire Feature [0..34]", "Encoded Value", "Normalized [0..1]"], empire_rows))
print("\nCold-Start vs Complete-History Feature Comparison (Indices 35..45)")
print(render_table(["History Feature", "Cold-Start", "Full-History", "Delta", "|Delta| Bar"], feat_cmp_rows))

## 3B. Deterministic Forward Rollout Simulator & Online MLP Value Network

Before running evolutionary search, let us inspect the two evaluation engines side by side:

1. **Deterministic Horizon Rollout (`simulate_action_sequence`)**: Steps through a candidate sequence of `SimAction` items (`BUILD`, `RESEARCH`, `BUILD_SHIP`, `ATTACK`, `DEMOLISH_BUILDING`, etc.), advancing passive mine production during resource waits, enforcing energy/field/bash rules, and scoring points, hourly `SpiceEq` income, and expected raid plunder.
2. **Online Neural Value Network (`create_value_network`)**: A two-hidden-layer MLP (`46 -> 32 -> 16 -> 1`, `TorchMLP` when PyTorch is installed or `PurePythonMLP` fallback) trained online on terminal rollout vectors with bounded target $y = \tanh(\text{score} / 25)$.
3. **Input Sensitivities ($\partial V / \partial x_j$)**: Reveals which encoded empire/history dimensions exert the strongest leverage on predicted state value, cross-checked against exact PyTorch autograd when available.

In [ ]:
eval_cfg = StrategyConfig(
    planner_mode="ga_nn",
    playstyle="fleeter",
    allow_raids=True,
    allow_shipyard=True,
    allow_research=True,
)
candidate_rollouts: list[tuple[str, list[SimAction]]] = [
    ("Idle baseline", []),
    ("Pig mine + Fusion", [SimAction("BUILD", 12476, 1), SimAction("BUILD", 12476, 5)]),
    ("Research Energy + Krypt mine", [SimAction("RESEARCH", 12476, "r2"), SimAction("BUILD", 12476, 3)]),
    ("Shipyard fighters", [SimAction("BUILD_SHIP", 12476, 4, quantity=3)]),
    ("Raid Trap 02:380:05 (10% win)", [SimAction("ATTACK", 12476, target_coords="02:380:05")]),
    ("Raid Farm 02:380:04 (90% win)", [SimAction("ATTACK", 12476, target_coords="02:380:04")]),
    (
        "Combo: Farm + Mine + Fighter",
        [
            SimAction("ATTACK", 12476, target_coords="02:380:04"),
            SimAction("BUILD", 12476, 1),
            SimAction("BUILD_SHIP", 12476, 4, quantity=2),
        ],
    ),
]

rollout_samples: list[tuple[list[float], float]] = []
rollout_records: list[tuple[str, Any]] = []
for label, seq in candidate_rollouts:
    sim_res = simulate_action_sequence(exp_state, seq, config=eval_cfg, raid_stats=demo_stats)
    term_vec = encode_empire_state(sim_res.final_state, raid_stats=demo_stats)
    target_val = math.tanh(sim_res.composite_growth_score / 25.0)
    rollout_samples.append((term_vec, target_val))
    rollout_records.append((label, sim_res))

max_rollout_score = max(
    (max(0.0, r.composite_growth_score) for _, r in rollout_records),
    default=1.0,
)
rollout_table_rows: list[list[Any]] = [
    [
        label,
        f"{sim_res.valid_steps}/{len(seq)}",
        f"{sim_res.points_delta:+.2f}",
        f"{sim_res.hourly_spice_eq_delta:+.1f}",
        f"{sim_res.raid_fleet_output_delta:+.2f}",
        f"{sim_res.composite_growth_score:.2f}",
        ascii_bar(max(0.0, sim_res.composite_growth_score), max(1e-6, max_rollout_score), width=12),
    ]
    for (label, seq), (_, sim_res) in zip(candidate_rollouts, rollout_records)
]

print(render_banner("3B. Forward Simulation Rollout Candidate Scores"))
print(
    render_table(
        ["Sequence", "Valid", "Pts Delta", "Income SpEq/h", "Raid/Fleet Delta", "Composite", "Score Bar"],
        rollout_table_rows,
    )
)

nn_net = create_value_network(backend="auto", seed=42)
pre_cold_v = nn_net.predict(vec_cold)
pre_full_v = nn_net.predict(vec_full)
train_stats = nn_net.train_on_rollouts(rollout_samples, epochs=12, lr=0.025, weight_decay=1e-4)
post_cold_v = nn_net.predict(vec_cold)
post_full_v = nn_net.predict(vec_full)

eps = 1e-4
sensitivities: list[tuple[str, float, float]] = []
for j, fname in enumerate(FEATURE_NAMES):
    xp, xm = list(vec_full), list(vec_full)
    xp[j] += eps
    xm[j] -= eps
    grad_j = (nn_net.predict(xp) - nn_net.predict(xm)) / (2.0 * eps)
    sensitivities.append((fname, vec_full[j], grad_j))

if TORCH_AVAILABLE and torch is not None and nn_net.backend == "torch":
    x_t = torch.tensor(vec_full, dtype=torch.float32, requires_grad=True)
    y_t = nn_net.impl.forward(x_t).squeeze()
    y_t.backward()
    torch_grads = [float(g) for g in x_t.grad.detach().tolist()]
    max_grad_diff = max(abs(a[2] - b) for a, b in zip(sensitivities, torch_grads))
else:
    max_grad_diff = 0.0

top_sens = sorted(sensitivities, key=lambda item: abs(item[2]), reverse=True)[:10]
max_abs_grad = max((abs(g) for _, _, g in top_sens), default=1.0)
sens_rows = [
    [fname, f"{xval:.4f}", f"{grad:+.5f}", ascii_bar(abs(grad), max_abs_grad, width=14)]
    for fname, xval, grad in top_sens
]

print("\nNeural Value Network Training & Top Input Sensitivities")
print(
    render_table(
        ["NN Metric", "Value"],
        [
            ["Backend / Parameters", f"{nn_net.backend} / {nn_net.parameter_count():,} params"],
            ["Rollout Loss (Initial -> Final)", f"{train_stats.initial_loss:.6f} -> {train_stats.final_loss:.6f}"],
            ["Weight Delta Norm", f"{train_stats.weight_delta_norm:.6f}"],
            ["Pred Value Cold -> Full (Pre / Post)", f"({pre_cold_v:+.4f} -> {pre_full_v:+.4f}) / ({post_cold_v:+.4f} -> {post_full_v:+.4f})"],
            ["FD vs Autograd Max Diff", f"{max_grad_diff:.2e}"],
        ],
    )
)
print(render_table(["Feature", "Value x_j", "dV/dx_j", "Magnitude"], sens_rows))

## 4A. Genetic Algorithm + Neural Network (`ga_nn`) Evolution Loop

`run_optimizer(...)` evolves a population of fixed-horizon `SimAction` chromosomes over `G` generations:

- **Seeded + Diverse Population**: Chromosome `0` encodes the heuristic `priority` baseline so elitism guarantees `best_fitness >= baseline_fitness`, while remaining chromosomes explore diverse mine, tech, shipyard, and raid combinations.
- **Blended Fitness Objective**: Combines deterministic rollout growth with online `NeuralValueNetwork` terminal-state guidance controlled by `ga_nn_weight`.
- **Online Value-Net Refinement**: After each generation's rollout batch, `NeuralValueNetwork` updates its weights on observed `(terminal_features, tanh_score)` pairs (`nn_loss` decreasing across generations).

In [ ]:
prio_cfg = StrategyConfig(
    planner_mode="priority",
    playstyle="fleeter",
    allow_raids=True,
    allow_shipyard=True,
    allow_research=True,
)
ga_cfg = StrategyConfig(
    planner_mode="ga_nn",
    playstyle="fleeter",
    allow_raids=True,
    allow_shipyard=True,
    allow_research=True,
    ga_population_size=16,
    ga_generations=8,
    ga_horizon_steps=5,
    ga_seed=42,
)

summary = run_optimizer(exp_state, config=ga_cfg, raid_stats=demo_stats)
direct_opt_actions = optimize_plan_actions(exp_state, config=ga_cfg, raid_stats=demo_stats)
prio_actions = plan_actions(exp_state, config=prio_cfg, raid_stats=demo_stats)
ga_actions = plan_actions(exp_state, config=ga_cfg, raid_stats=demo_stats)
disk_only_actions = optimize_plan_actions(
    base_state_no_hist, config=ga_cfg, raid_stats=disk_attack_history
)
eta_plans: dict[str, list[PlannedAction]] = {
    "priority": prio_actions,
    "ga_nn": ga_actions,
    "disk_history_ga_nn": attach_action_etas(base_state_no_hist, disk_only_actions),
}

print(render_banner("4A. GA + NN Optimizer Run Summary"))
print(
    render_table(
        ["Metric", "Value"],
        [
            ["NN Backend / Evaluations", f"{summary.nn_backend} / {summary.nn_evaluations} evals"],
            ["Population / Generations / Seed", f"{summary.population_size} / {summary.generations} / {summary.seed}"],
            ["Baseline -> Best Fitness", f"{summary.baseline_fitness:.3f} -> {summary.best_fitness:.3f} ({summary.improvement_pct:+.2f}%)"],
            ["NN Loss (Initial -> Final)", f"{summary.nn_initial_loss:.6f} -> {summary.nn_final_loss:.6f} (delta={summary.nn_weight_delta_norm:.4f})"],
            ["Best Sim Deltas (Pts / Inc / Raid)", f"{summary.best_simulation.points_delta:+.2f} pts / {summary.best_simulation.hourly_spice_eq_delta:+.1f} SpEq/h / {summary.best_simulation.raid_fleet_output_delta:+.2f}"],
            ["Elapsed Time", f"{summary.elapsed_ms:.2f} ms"],
        ],
    )
)

gen_max_fit = max((g.best_fitness for g in summary.generation_history), default=1.0)
gen_rows = [
    [
        g.generation,
        f"{g.best_fitness:.3f}",
        f"{g.mean_fitness:.3f}",
        f"{g.worst_fitness:.3f}",
        f"{g.best_points_gain:+.2f}",
        f"{g.best_income_gain:+.1f}",
        f"{g.best_raid_value:+.2f}",
        f"{g.nn_loss:.5f}",
        ascii_bar(max(0.0, g.best_fitness), max(1e-6, gen_max_fit), width=14),
    ]
    for g in summary.generation_history
]
print("\nGeneration-by-Generation Convergence")
print(
    render_table(
        ["Gen", "Best Fit", "Mean Fit", "Worst Fit", "Pts Gain", "Inc Gain", "Raid Val", "NN Loss", "Curve"],
        gen_rows,
    )
)

if HAS_MATPLOTLIB and plt is not None and summary.generation_history:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    try:
        gens_x = [g.generation for g in summary.generation_history]
        ax.plot(gens_x, [g.best_fitness for g in summary.generation_history], label="Best Fitness")
        ax.plot(gens_x, [g.mean_fitness for g in summary.generation_history], label="Mean Fitness")
        ax.set_xlabel("Generation")
        ax.set_ylabel("Fitness")
        ax.set_title("GA + NN Fitness Progression")
        ax.legend()
        get_be = getattr(matplotlib, "get_backend", None) if matplotlib is not None else None
        backend_name = str(get_be()).lower() if callable(get_be) else ""
        if "inline" in backend_name and hasattr(plt, "show"):
            plt.show()
    finally:
        plt.close(fig)

## 4B. Best Chromosome Rollout Trace, Plan Comparison & Queue ETAs

Once `run_optimizer(...)` selects its winning chromosome, we can inspect:

1. **Step-by-Step Horizon Rollout (`best_simulation.step_outcomes`)**: Every simulated gene (`#0..#H-1`), whether it executed legally, how long passive mines accumulated resources before execution, and what build/raid outcome resulted.
2. **Heuristic (`"priority"`) vs Optimizer (`"ga_nn"`) Queue ETAs (`attach_action_etas`)**: Compares immediate executable actions (`Now=Y`), resource-wait times (`Wait ETA`), construction/research durations (`Duration`), and bottleneck resources (`pig_iron`, `metal`, `kryptonite`, `spice`).
3. **Anti-Trap Raid Selection**: Confirms `ga_nn` prioritizes **`02:380:04 (TargetAlpha)`** (`90%` historical win rate) and rejects the `02:380:05 (TargetBeta)` `10%` fluke trap despite `02:380:05` appearing first in `raid_targets`!

In [ ]:
summary_dict = summary.to_dict()
step_outcomes = _as_list(_as_dict(summary_dict.get("best_simulation")).get("step_outcomes"))
step_trace_rows: list[list[Any]] = []
for step_idx, raw_step in enumerate(step_outcomes):
    st_d = _as_dict(raw_step)
    step_trace_rows.append(
        [
            st_d.get("index", step_idx),
            st_d.get("kind", "-"),
            st_d.get("target_coords") or st_d.get("target_id") or "-",
            st_d.get("quantity", 1),
            "Y" if st_d.get("valid", True) else "N",
            f"{_safe_float(st_d.get('wait_hours', 0.0)):.2f}h",
            redact_sensitive(str(st_d.get("note") or st_d.get("status") or "Simulated")),
        ]
    )

print(render_banner("4B. Winning Chromosome Horizon Rollout & Queue ETAs"))
if step_trace_rows:
    print("Winning Chromosome Step-by-Step Horizon Trace")
    print(
        render_table(
            ["Gene #", "Kind", "Target", "Qty", "Valid", "Sim Wait", "Outcome Note"],
            step_trace_rows,
        )
    )


def _plan_rows(label: str, actions: list[PlannedAction], limit: int = 8) -> list[list[Any]]:
    rows: list[list[Any]] = []
    for idx, act in enumerate(actions[:limit], start=1):
        rows.append(
            [
                label,
                idx,
                act.priority,
                act.kind,
                act.target_name,
                "Y" if act.executable_now else "N",
                act.wait_eta,
                act.duration_text,
                act.total_eta,
                act.eta_bottleneck or "-",
            ]
        )
    return rows


cmp_rows = _plan_rows("priority", prio_actions, limit=6) + _plan_rows("ga_nn", ga_actions, limit=8)
print("\nBaseline ('priority') vs Optimizer ('ga_nn') Action Plan & Queue ETAs")
print(
    render_table(
        ["Mode", "#", "Prio", "Kind", "Target", "Now", "Wait ETA", "Duration", "Total ETA", "Bottleneck"],
        cmp_rows,
    )
)

ga_attack_targets = [a.target_name for a in ga_actions if a.kind == "ATTACK"]
disk_attack_targets = [a.target_name for a in disk_only_actions if a.kind == "ATTACK"]
print(f"GA+NN prioritized attack targets (in execution order): {ga_attack_targets}")
print(
    f"Direct optimize_plan_actions count={len(direct_opt_actions)}, "
    f"disk-only fallback count={len(disk_only_actions)}, "
    f"disk-only attack targets={disk_attack_targets}"
)

bottleneck_probe = PlannedAction(
    priority="P2",
    kind="BUILD",
    planet_id=12476,
    planet_coords="02:380:12",
    target_id=11,
    target_name="Pig iron warehouse",
    target_level=14,
    cost=Resources(50_000, 15_000, 20_000, 5_000),
    missing=Resources(41_000, 8_000, 15_500, 2_800),
    reason="Resource-wait bottleneck ETA inspection",
    executable_now=False,
)
probe_eta = compute_action_eta(exp_state, bottleneck_probe)
print(
    f"Sample resource-wait ETA (Pig iron warehouse L14): wait={probe_eta['wait_eta']}, "
    f"duration={probe_eta['duration_text']}, total={probe_eta['total_eta']}, "
    f"bottleneck={probe_eta['eta_bottleneck']}"
)

## 5. Topological Action Graph (DAG) & Isolated Parameter Override Preview

The Operator Control UI's **Action Graph** tab (`[data-page="action-graph"]`) is powered by `spaceinvasion.action_graph`:

- **Kahn Longest-Path Topological Layering (`build_action_graph`)**: Organizes planned actions into colony + empire lanes with strict `layer[source] < layer[target]` acyclicity across 11 typed directed edge classes (`slot_order`, `ga_sequence`, `resource_feed`, `energy_supply`, `storage_cap`, `prereq_building`, `prereq_tech`, `recon_intel`, `fleet_prep`, `hostile_defense`, `tutorial_step`).
- **Ephemeral Parameter Simulation (`merge_strategy_config_overrides`)**: Computes `(effective_config, changed_keys)` and runs `run_optimizer(..., record_global_summary=False)` in isolation so previewing parameter edits never mutates the active `StrategyConfig` or global optimizer summary.

In [ ]:
prio_graph = build_action_graph(
    exp_state,
    prio_actions,
    effective_config=prio_cfg,
    active_config=prio_cfg,
    simulated=False,
)
ga_graph = build_action_graph(
    exp_state,
    ga_actions,
    effective_config=ga_cfg,
    active_config=ga_cfg,
    simulated=False,
    optimizer_summary=summary_dict,
)

# Preview an isolated parameter change (switching fleeter -> miner, disabling raids) without mutating ga_cfg.
sim_preview_cfg, sim_changed_keys = merge_strategy_config_overrides(
    ga_cfg,
    {"playstyle": "miner", "allow_raids": False, "ga_nn_weight": 0.50},
)
sim_preview_opt = run_optimizer(
    exp_state,
    config=sim_preview_cfg,
    raid_stats=demo_stats,
    record_global_summary=False,
)
sim_preview_actions = attach_action_etas(exp_state, sim_preview_opt.actions)
sim_preview_graph = build_action_graph(
    exp_state,
    sim_preview_actions,
    effective_config=sim_preview_cfg,
    active_config=ga_cfg,
    simulated=True,
    optimizer_summary=sim_preview_opt.to_dict(),
)

assert ga_cfg.playstyle == "fleeter" and ga_cfg.allow_raids is True
assert sim_preview_graph["is_dirty"] is True

graph_cmp_rows: list[list[Any]] = []
for g_label, g_obj in (
    ("Active Heuristic (priority)", prio_graph),
    ("Active Optimizer (ga_nn)", ga_graph),
    ("Isolated Preview (miner, no-raids)", sim_preview_graph),
):
    g_sum = _as_dict(g_obj.get("summary"))
    edge_counts = ", ".join(
        f"{k}:{v}" for k, v in sorted(_as_dict(g_sum.get("by_edge_type")).items())
    ) or "-"
    graph_cmp_rows.append(
        [
            g_label,
            g_obj.get("mode"),
            "Y" if g_obj.get("is_dirty") else "N",
            f"{g_sum.get('total_nodes', 0)} / {g_sum.get('total_edges', 0)}",
            f"L0..L{g_sum.get('max_layer', 0)}",
            f"{g_sum.get('ready_count', 0)}R / {g_sum.get('waiting_count', 0)}W",
            g_sum.get("critical_path_eta", "-"),
            edge_counts,
        ]
    )

print(render_banner("5. Topological Action Graph (DAG) & Isolated Preview"))
print(
    render_table(
        ["Graph Variant", "Mode", "Dirty", "Nodes/Edges", "Layers", "Ready/Wait", "Crit ETA", "Edge Types"],
        graph_cmp_rows,
    )
)
print(f"Preview changed_keys (active ga_cfg untouched): {sim_changed_keys}")

node_by_id = {str(n.get("id")): _as_dict(n) for n in _as_list(ga_graph.get("nodes"))}
edge_preview_rows: list[list[Any]] = []
for edge in _as_list(ga_graph.get("edges"))[:10]:
    ed = _as_dict(edge)
    src_n = _as_dict(node_by_id.get(str(ed.get("source", ""))))
    dst_n = _as_dict(node_by_id.get(str(ed.get("target", ""))))
    edge_preview_rows.append(
        [
            f"L{src_n.get('topological_layer', '?')} -> L{dst_n.get('topological_layer', '?')}",
            ed.get("primary_type", "-"),
            f"{src_n.get('kind', '?')}:{src_n.get('target_name', ed.get('source', '?'))}",
            f"{dst_n.get('kind', '?')}:{dst_n.get('target_name', ed.get('target', '?'))}",
            redact_sensitive(str(ed.get("label", ""))),
        ]
    )

print("\nDirected Dependency & Sequence Edges in 'ga_nn' Action Graph (First 10)")
print(
    render_table(
        ["Layer Hop", "Edge Class", "Source Node", "Target Node", "Dependency Reason"],
        edge_preview_rows,
    )
)

## 6. Strategic Policy Ablation: Planetary Solar Ban (`DEMOLISH_BUILDING`) & Field Economics

In SpaceInvasion Classic, building fields on normal colonies are finite:

- **`Thermosolar plant` (`b6`)** generates ~20% less power per building field than **`Fusion plant` (`b5`)**, burns Spice continuously, and collapses to `0` output if Spice hits `0`.
- When `StrategyConfig(forbid_planetary_solar=True)` is active on a normal colony that still has legacy `b6 > 0` (and sufficient `b5` power headroom), both `"priority"` and `"ga_nn"` schedule **`DEMOLISH_BUILDING`** (`b6 -> b6 - 1`) to reclaim colony building fields while reserving solar strictly for Asteroids.

In [ ]:
legacy_solar_state = build_sample_game_state(
    timestamp=base_ts,
    buildings={
        "b1": 7,
        "b2": 5,
        "b3": 6,
        "b4": 4,
        "b5": 11,
        "b6": 3,
        "b7": 3,
        "b8": 2,
        "b9": 3,
        "b11": 3,
        "b12": 3,
        "b13": 3,
        "b14": 3,
    },
    include_raid_targets=False,
)

solar_ablation_rows: list[list[Any]] = []
solar_demolish_graph: dict[str, Any] = {}
for label, mode_name, forbid_solar in (
    ("Heuristic (allow solar)", "priority", False),
    ("Heuristic (forbid solar)", "priority", True),
    ("GA + NN (allow solar)", "ga_nn", False),
    ("GA + NN (forbid solar)", "ga_nn", True),
):
    ab_cfg = StrategyConfig(
        planner_mode=mode_name,
        playstyle="miner",
        allow_raids=False,
        forbid_planetary_solar=forbid_solar,
        ga_population_size=12,
        ga_generations=5,
        ga_horizon_steps=4,
        ga_seed=42,
    )
    if mode_name == "ga_nn":
        ab_opt = run_optimizer(
            legacy_solar_state,
            config=ab_cfg,
            raid_stats=demo_stats,
            record_global_summary=False,
        )
        ab_acts = attach_action_etas(legacy_solar_state, ab_opt.actions)
        ab_sum_d = ab_opt.to_dict()
    else:
        ab_acts = plan_actions(legacy_solar_state, config=ab_cfg, raid_stats=demo_stats)
        ab_sum_d = None

    has_demolish = any(a.kind == "DEMOLISH_BUILDING" for a in ab_acts)
    top3 = ", ".join(f"{a.kind}:{a.target_name}" for a in ab_acts[:3]) or "NONE"
    solar_ablation_rows.append(
        [
            label,
            mode_name,
            "Y" if forbid_solar else "N",
            "YES (reclaims b6)" if has_demolish else "No",
            top3,
        ]
    )
    if mode_name == "ga_nn" and forbid_solar:
        solar_demolish_graph = build_action_graph(
            legacy_solar_state,
            ab_acts,
            effective_config=ab_cfg,
            active_config=ab_cfg,
            simulated=True,
            optimizer_summary=ab_sum_d,
        )

print(render_banner("6. Planetary Solar Policy Ablation (Legacy b6=3 Colony)"))
print(
    render_table(
        ["Policy Variant", "Mode", "Forbid Solar", "Demolishes b6?", "Top-3 Planned Actions"],
        solar_ablation_rows,
    )
)
demolish_nodes = [
    f"{n.get('kind')} -> {n.get('target_name')} (L{n.get('topological_layer')}, status={n.get('status')})"
    for n in _as_list(solar_demolish_graph.get("nodes"))
    if _as_dict(n).get("kind") == "DEMOLISH_BUILDING"
]
print(f"Action Graph DEMOLISH_BUILDING nodes under GA+NN solar ban: {demolish_nodes}")

## 7. Interactive Hyperparameter Tuning Grid & Playstyle Benchmark

Edit `EXPERIMENT_CONFIGS` below to explore how `StrategyConfig` knobs reshape the winning plan, fitness gain, and runtime latency:

| Parameter | Role in `ga_nn` | Recommended Range |
|---|---|---|
| `playstyle` | Weights economy growth vs raid plunder (`"miner"`, `"hybrid"`, `"fleeter"`) | `"miner"` / `"hybrid"` / `"fleeter"` |
| `ga_population_size` | Number of candidate action chromosomes evaluated per generation | `12 .. 48` |
| `ga_generations` | Number of evolutionary + online NN update rounds | `5 .. 20` |
| `ga_horizon_steps` | Simulated lookahead length (`SimAction` genes per chromosome) | `3 .. 8` |
| `ga_nn_weight` | Blend weight on `NeuralValueNetwork` terminal-state prediction | `0.00 .. 0.65` |
| `ga_mutation_rate` | Per-gene mutation rate balancing exploration vs exploitation | `0.15 .. 0.45` |
| `ga_seed` | Deterministic RNG seed (`None` for non-deterministic runtime jitter) | `int` or `None` |

In [ ]:
EXPERIMENT_CONFIGS: list[dict[str, Any]] = [
    {
        "label": "miner_default",
        "playstyle": "miner",
        "ga_population_size": 12,
        "ga_generations": 5,
        "ga_horizon_steps": 4,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.25,
        "ga_seed": 42,
    },
    {
        "label": "hybrid_balanced",
        "playstyle": "hybrid",
        "ga_population_size": 12,
        "ga_generations": 5,
        "ga_horizon_steps": 4,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.25,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_pure_ga",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.0,
        "ga_mutation_rate": 0.35,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_nn_035",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.35,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_nn_high_mut",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.65,
        "ga_mutation_rate": 0.40,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_seed_7",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.35,
        "ga_seed": 7,
    },
]

sweep_results: list[dict[str, Any]] = []
for spec in EXPERIMENT_CONFIGS:
    sw_cfg = StrategyConfig(
        planner_mode="ga_nn",
        playstyle=str(spec["playstyle"]),
        allow_raids=True,
        allow_shipyard=True,
        allow_research=True,
        ga_population_size=int(spec["ga_population_size"]),
        ga_generations=int(spec["ga_generations"]),
        ga_horizon_steps=int(spec["ga_horizon_steps"]),
        ga_nn_weight=float(spec["ga_nn_weight"]),
        ga_mutation_rate=float(spec["ga_mutation_rate"]),
        ga_seed=int(spec["ga_seed"]),
    )
    sw_sum = run_optimizer(
        exp_state,
        config=sw_cfg,
        raid_stats=demo_stats,
        record_global_summary=False,
    )
    sw_eta = attach_action_etas(exp_state, sw_sum.actions)
    top_act = sw_eta[0] if sw_eta else None
    top_label = f"{top_act.kind}:{top_act.target_name}" if top_act is not None else "NONE"
    top_total_eta = top_act.total_eta if top_act is not None else "-"
    sweep_results.append(
        {
            "label": spec["label"],
            "playstyle": sw_cfg.playstyle,
            "population_size": sw_sum.population_size,
            "generations": sw_sum.generations,
            "horizon_steps": sw_cfg.ga_horizon_steps,
            "nn_weight": sw_cfg.ga_nn_weight,
            "mutation_rate": sw_cfg.ga_mutation_rate,
            "seed": sw_sum.seed,
            "baseline_fitness": sw_sum.baseline_fitness,
            "best_fitness": sw_sum.best_fitness,
            "improvement_pct": sw_sum.improvement_pct,
            "raid_fleet_delta": sw_sum.best_simulation.raid_fleet_output_delta,
            "income_delta": sw_sum.best_simulation.hourly_spice_eq_delta,
            "points_delta": sw_sum.best_simulation.points_delta,
            "top_action": top_label,
            "top_total_eta": top_total_eta,
            "elapsed_ms": sw_sum.elapsed_ms,
        }
    )

max_sweep_fit = max((max(0.0, r["best_fitness"]) for r in sweep_results), default=1.0)
sweep_rows: list[list[Any]] = [
    [
        rec["label"],
        rec["playstyle"],
        f"{rec['population_size']}/{rec['generations']}/{rec['horizon_steps']}",
        f"{rec['nn_weight']:.2f}/{rec['mutation_rate']:.2f}/{rec['seed']}",
        f"{rec['baseline_fitness']:.2f} -> {rec['best_fitness']:.2f}",
        f"{rec['improvement_pct']:+.2f}%",
        f"{rec['raid_fleet_delta']:+.2f}",
        f"{rec['top_action']} ({rec['top_total_eta']})",
        ascii_bar(max(0.0, rec["best_fitness"]), max(1e-6, max_sweep_fit), width=10),
        f"{rec['elapsed_ms']:.1f}",
    ]
    for rec in sweep_results
]

best_sweep = max(sweep_results, key=lambda r: float(r["best_fitness"]))
print(render_banner("7. Hyperparameter Sweep Leaderboard"))
print(
    render_table(
        ["Config", "Style", "Pop/Gen/Hor", "NNw/Mut/Seed", "Base -> Best Fit", "Improv", "Raid/Fleet", "Top Action (Total ETA)", "Fit Bar", "ms"],
        sweep_rows,
    )
)
print(
    f"Best overall sweep configuration: {best_sweep['label']} "
    f"(best_fitness={best_sweep['best_fitness']:.3f}, {best_sweep['improvement_pct']:+.2f}%, "
    f"top_action={best_sweep['top_action']})"
)

## 8. Operator Quick-Reference & Live Control UI Mapping

After experimenting with configurations in this notebook, you can preview or persist your chosen settings directly in the live **`SpaceInvasion Control`** web console (`http://127.0.0.1:8787/`):

1. **Open the `Action Graph` Tab (`[data-page="action-graph"]`)**:
   - Switch between **`Heuristic (priority)`** and **`GA + Neural (ga_nn)`** at the top of the graph canvas to compare directed dependency flows (`L0 -> Lk`) live.
2. **Simulate Parameters in the Right-Side Control Panel (`#ag-control-panel`)**:
   - Adjust `playstyle`, `ga_population_size`, `ga_generations`, `ga_horizon_steps`, `ga_nn_weight`, `ga_mutation_rate`, `forbid_planetary_solar`, or `min_colony_fields` — every edit runs an isolated preview (`POST /command {"command": "action_graph"}`) with **zero browser page views** and **zero active config mutation**.
3. **Persist When Satisfied**:
   - Click **`Apply to Active Config`** in the right-side panel (or edit `config.yaml` directly based on `config.example.yaml`) to save your tuned parameters to the active autopilot.